In [25]:
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
# xx_monthly_users.py
# Purpose of Script: Process Table of Monthly Users (Sourced from External
# Reports & Saved in Excel).
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
# Initialization ----
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
# Import Google Drive
#~~~~~~~~~~~~~~~~~~~~~~~~~~
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [60]:
#~~~~~~~~~~~~~~~~~~~~~~~~~~
# Import Libraries
#~~~~~~~~~~~~~~~~~~~~~~~~~~
import numpy as np
import pandas as pd

In [61]:
#~~~~~~~~~~~~~~~~~~~~~~~~~~
# Define Input/Output Paths
#~~~~~~~~~~~~~~~~~~~~~~~~~~
path_users = "/content/drive/MyDrive/Colab Notebooks/hsds/xx_dissertation/01_final_data/04_supporting/"

In [62]:
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
# Import Data ----
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
df = pd.read_excel(f"{path_users}xx_average_monthly_users.xlsx")

In [63]:
df

,platform,start_date,end_date,country,country_code,users
0,x,2024-10-01,2025-03-31,austria,AT,1347884
1,x,2024-10-01,2025-03-31,belgium,BE,2266281
2,x,2024-10-01,2025-03-31,bulgaria,BG,664442
3,x,2024-10-01,2025-03-31,croatia,HR,276537
4,x,2024-10-01,2025-03-31,cyprus,CY,1890675
...,...,...,...,...,...,...
530,youtube,2026-01-01,2026-05-31,sweden,SE,30100000
531,youtube,2026-01-01,2026-05-31,all,EU27,1230440000
532,whatsapp,2025-01-01,2025-06-30,all,EU27,51700000
533,whatsapp,2025-07-01,2025-12-31,all,EU27,58300000


In [67]:
# Expand to form daily data
df_exp = df.copy()
df_exp["start_date"] = pd.to_datetime(df_exp["start_date"])
df_exp["end_date"] = pd.to_datetime(df_exp["end_date"])

# Find Min and Max Dates
min_start = df_exp["start_date"].min()
max_end = df_exp["end_date"].max()
sample_range = pd.date_range(min_start, max_end, freq="D")

# Isolate Platforms
plats = df_exp["platform"].unique()

# Build Cross Table
df_users = pd.MultiIndex.from_product([plats, sample_range], names=["platform","date"]).to_frame(index=False)

# Filter Data Table
df_cut = df[df["country"] == "all"]
df_cut = df_cut[["platform","start_date","end_date","users"]]

# Join Original Data Table
df_users = df_users.merge(df_cut, on="platform", how="left")

# Filter User Data for Correct Date Range
df_users = df_users[(df_users["date"] >= df_users["start_date"]) &
                    (df_users["date"] <= df_users["end_date"])]

df_users = df_users.drop(columns= ["start_date","end_date"])
df_users = df_users[df_users["date"] >= "2025-01-01"]

In [68]:
df_users

,platform,date,users
368,x,2025-01-01,94830300
372,x,2025-01-02,94830300
376,x,2025-01-03,94830300
380,x,2025-01-04,94830300
384,x,2025-01-05,94830300
...,...,...,...
13363,whatsapp,2026-05-27,58300000
13366,whatsapp,2026-05-28,58300000
13369,whatsapp,2026-05-29,58300000
13372,whatsapp,2026-05-30,58300000


In [66]:
df_users.groupby("platform").size()

,0
platform,
facebook,516
instagram,516
snapchat,516
tiktok,516
whatsapp,516
x,608
youtube,516


In [59]:
df_users.groupby("platform").agg(
    first_day=("date", "min"),
    last_day=("date", "max"),
    n_days=("date", "count")
)

,first_day,last_day,n_days
platform,,,
facebook,2025-01-01,2026-05-31,516
instagram,2025-01-01,2026-05-31,516
snapchat,2025-01-01,2026-05-31,516
tiktok,2025-01-01,2026-05-31,516
whatsapp,2025-01-01,2026-06-30,546
x,2024-10-01,2026-05-31,608
youtube,2025-01-01,2026-05-31,516
